# Conditional Conformal Selection

This notebook mirrors the conditional conformal script example and compares `Empirical()` against `ConditionalEmpirical()`.

## Import

This section loads all dependencies used throughout the notebook.

In [1]:
import logging

import pandas as pd
from oddball import Dataset, load
from pyod.models.iforest import IForest

from nonconform import ConformalDetector, Empirical, Split
from nonconform.metrics import false_discovery_rate, statistical_power
from nonconform.scoring import ConditionalEmpirical

root_logger = logging.getLogger("nonconform")
if not root_logger.handlers:
    root_logger.addHandler(logging.NullHandler())
root_logger.setLevel(logging.ERROR)

## Setup

We load Shuttle with a fixed seed and keep detector and strategy constant so only the estimation method changes.

In [2]:
x_train, x_test, y_test = load(Dataset.SHUTTLE, setup=True, seed=1)

n_calib = 1_000
alpha = 0.2

print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")
print(f"y_test positives: {int(y_test.sum())}")
print(f"alpha={alpha}, calibration size={n_calib}")

x_train: (22793, 9), x_test: (1000, 9)
y_test positives: 100
alpha=0.2, calibration size=1000


## Estimation Comparison

Both rows use `detector.select()`, while the p-value estimator changes from standard empirical to conditional empirical calibration.

In [3]:
def summarize_row(name, decisions, p_values, y_true):
    """Compute summary metrics for one estimator configuration."""
    return {
        "method": name,
        "discoveries": int(decisions.sum()),
        "p_min": float(p_values.min()),
        "p_max": float(p_values.max()),
        "fdr": float(false_discovery_rate(y=y_true, y_hat=decisions)),
        "power": float(statistical_power(y=y_true, y_hat=decisions)),
    }


rows = []
configs = [
    ("Empirical", Empirical()),
    (
        "ConditionalEmpirical (simes, delta=0.1)",
        ConditionalEmpirical(method="simes", delta=0.1),
    ),
]

for name, estimator in configs:
    detector = ConformalDetector(
        detector=IForest(),
        strategy=Split(n_calib=n_calib),
        estimation=estimator,
        seed=1,
    )

    detector.fit(x_train)
    decisions = detector.select(x_test, alpha=alpha)
    p_values = detector.last_result.p_values
    rows.append(summarize_row(name, decisions, p_values, y_test))

results = pd.DataFrame(rows)
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

                                 method  discoveries  p_min  p_max    fdr  power
                              Empirical          132 0.0010 1.0000 0.2500 0.9900
ConditionalEmpirical (simes, delta=0.1)          121 0.0046 1.0000 0.1901 0.9800


## Interpretation

`ConditionalEmpirical()` applies an additional finite-sample conditional calibration map to the base empirical conformal p-values before selection.